# 01 — Data Exploration & Feature Engineering

This notebook explores the 2025 NFL play-by-play data and validates the QB feature engineering pipeline.

**Goals:**
- Understand the shape and quality of our data
- Visualize distributions of key passing metrics
- Build and inspect the QB feature matrix
- Sanity-check metrics against known QB archetypes

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Load the Data

In [ ]:
df = pd.read_parquet('../data/processed/pass_plays_qualified.parquet')
print(f'Shape: {df.shape}')
print(f'QBs: {df.passer_player_name.nunique()}')
print(f'Weeks: {sorted(df.week.unique()) if "week" in df.columns else "N/A"}')
df.head()

In [ ]:
# Check for missing values in key columns
key_cols = ['air_yards', 'epa', 'complete_pass', 'sack', 'was_pressure',
            'score_differential', 'qb_scramble', 'pass_location']
df[key_cols].isna().sum().sort_values(ascending=False)

## 2. Play-Level Distributions

Let's get a feel for the data before aggregating to the QB level.

In [ ]:
# Attempts per QB
attempts = df.groupby('passer_player_name').size().sort_values(ascending=False).reset_index()
attempts.columns = ['QB', 'Attempts']

fig = px.bar(attempts, x='QB', y='Attempts',
             title='Pass Attempts by QB — 2025 Regular Season',
             color='Attempts', color_continuous_scale='blues')
fig.update_layout(xaxis_tickangle=-45, width=1000, height=500)
fig.show()

In [ ]:
# Air yards distribution
throws = df[df['sack'] == 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall air yards
axes[0].hist(throws['air_yards'].dropna(), bins=50, edgecolor='white', alpha=0.8)
axes[0].set_title('Air Yards Distribution (All Throws)')
axes[0].set_xlabel('Air Yards')
axes[0].axvline(throws['air_yards'].mean(), color='red', linestyle='--', label=f'Mean: {throws["air_yards"].mean():.1f}')
axes[0].legend()

# EPA distribution
axes[1].hist(throws['epa'].dropna(), bins=50, edgecolor='white', alpha=0.8, color='coral')
axes[1].set_title('EPA Distribution (All Throws)')
axes[1].set_xlabel('EPA')
axes[1].axvline(0, color='black', linestyle='-', linewidth=1)

# Completion rate by air yards
throws['air_yards_bin'] = pd.cut(throws['air_yards'], bins=range(-10, 55, 5))
comp_by_depth = throws.groupby('air_yards_bin')['complete_pass'].mean()
axes[2].plot(range(len(comp_by_depth)), comp_by_depth.values, marker='o', color='green')
axes[2].set_xticks(range(len(comp_by_depth)))
axes[2].set_xticklabels([str(x) for x in comp_by_depth.index], rotation=45)
axes[2].set_title('Completion Rate by Air Yards')
axes[2].set_ylabel('Completion %')

plt.tight_layout()
plt.show()

In [ ]:
# Pressure rate and its effect on EPA
pressure_avail = df['was_pressure'].notna().mean()
print(f'Pressure data available for {pressure_avail:.1%} of plays')

if pressure_avail > 0.5:
    pressure_effect = df[df['sack'] == 0].groupby('was_pressure')['epa'].mean()
    print(f'\nEPA clean pocket:    {pressure_effect.get(0, float("nan")):.3f}')
    print(f'EPA under pressure:  {pressure_effect.get(1, float("nan")):.3f}')
    print(f'Pressure penalty:    {pressure_effect.get(1, 0) - pressure_effect.get(0, 0):.3f} EPA/play')

## 3. Build the QB Feature Matrix

Now let's run our feature engineering pipeline and inspect the results.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.features.build_features import build_qb_features, get_clustering_features

features = build_qb_features(df)
print(f'Feature matrix: {features.shape}')
features.head(10)

In [ ]:
# Full feature summary
features.describe().round(3)

In [ ]:
# Top 10 QBs by key metrics
print('=== Most Aggressive (Avg Air Yards) ===')
print(features.nlargest(10, 'avg_air_yards')[['player_name', 'team', 'avg_air_yards', 'deep_ball_rate']].to_string(index=False))

print('\n=== Most Accurate (Completion %) ===')
print(features.nlargest(10, 'overall_comp_pct')[['player_name', 'team', 'overall_comp_pct', 'avg_air_yards']].to_string(index=False))

print('\n=== Best Under Pressure (Pressure Resilience) ===')
print(features.nlargest(10, 'pressure_resilience')[['player_name', 'team', 'pressure_resilience', 'epa_pressure']].to_string(index=False))

print('\n=== Most Clutch (Clutch EPA) ===')
print(features.nlargest(10, 'clutch_epa')[['player_name', 'team', 'clutch_epa', 'clutch_completion_pct']].to_string(index=False))

## 4. Feature Correlations

Check which features move together and which capture independent dimensions of play style.

In [ ]:
X_scaled, feature_cols = get_clustering_features(features)

fig, ax = plt.subplots(figsize=(14, 10))
corr = features[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Scatter Plot Previews

Some quick 2D views to build intuition before clustering.

In [ ]:
plot_df = features.reset_index()

fig = px.scatter(plot_df, x='avg_air_yards', y='overall_comp_pct',
                 text='player_name', size='pass_attempts',
                 color='avg_intended_epa',
                 color_continuous_scale='RdYlGn',
                 title='Aggression vs Accuracy (sized by attempts, colored by EPA)',
                 hover_data=['team'])
fig.update_traces(textposition='top center', textfont_size=8)
fig.update_layout(width=900, height=650)
fig.show()

In [ ]:
fig = px.scatter(plot_df, x='scramble_rate_mobility', y='avg_air_yards',
                 text='player_name', size='pass_attempts',
                 color='clutch_epa',
                 color_continuous_scale='RdYlGn',
                 title='Mobility vs Aggression (colored by Clutch EPA)',
                 hover_data=['team'])
fig.update_traces(textposition='top center', textfont_size=8)
fig.update_layout(width=900, height=650)
fig.show()

## 6. Quick Radar Chart Preview

Compare a couple of QBs to make sure the radar chart visualization works.

In [ ]:
from src.visualization.plots import plot_radar_comparison

# Find player IDs for a couple of interesting comparisons
name_to_id = features['player_name'].reset_index().set_index('player_name')['passer_player_id'].to_dict()

# Pick two QBs with contrasting styles — adjust if these aren't in your data
qb_pairs = [
    ('P.Mahomes', 'J.Goff'),
    ('L.Jackson', 'T.Tagovailoa'),
    ('J.Allen', 'B.Purdy'),
]

for qb1_name, qb2_name in qb_pairs:
    qb1_id = name_to_id.get(qb1_name)
    qb2_id = name_to_id.get(qb2_name)
    if qb1_id and qb2_id:
        fig = plot_radar_comparison(features, [qb1_id, qb2_id])
        fig.show()
        break  # Show one for now
    else:
        print(f'{qb1_name} or {qb2_name} not found, trying next pair...')

## 7. Save Feature Matrix

Save the engineered features for use in clustering and modeling notebooks.

In [ ]:
features.to_parquet('../data/processed/qb_features.parquet')
print(f'Saved QB feature matrix: {features.shape}')
print('\nReady for clustering in notebook 02!')

---

## Key Takeaways

*Fill this in after running the notebook:*

- How many QBs qualified?
- Which metrics show the most separation between QBs?
- Any surprising findings in the scatter plots?
- Which features are highly correlated (and might be redundant for clustering)?